# COMP64702 - RAG Culinary Assistant
## Notebook 03: Ingestion Pipeline
### =============================================================================
### This notebook covers:
####  (1) Chunking     — three strategies compared: fixed, sentence, semantic
####   (2) Embedding    — two models compared: MiniLM vs BGE
####   (3) Vector Store — FAISS index built and saved

#### The rubric awards marks for EXPLORING DIFFERENT TECHNIQUES so we
#### implement and compare multiple approaches for both chunking and embedding,
#### then select the best combination for the final vector store.
### =============================================================================
 

In [1]:
import os
import json
import glob
import pickle
import numpy as np
from datetime import datetime
from collections import Counter
 
os.chdir(r"D:\text mining\rag project")
print(f"Working directory: {os.getcwd()}")
 
os.makedirs("vector_store", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
 
# ── Install any missing packages ──────────────────────────────────────────────
# Run this in your terminal if needed:
# pip install sentence-transformers faiss-cpu nltk rank_bm25
 
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
 
print("Imports done")
 

Working directory: D:\text mining\rag project
Imports done


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Load all processed corpus documents
# ─────────────────────────────────────────────────────────────────────────────
 
def load_corpus():
    docs = []
    patterns = {
        "wikipedia": "data/raw/wikipedia_articles/*.json",
        "wikibooks": "data/raw/wikibooks_recipes/*.json",
        "blog":      "data/raw/blog_posts/*.json",
    }
    for source, pattern in patterns.items():
        for fp in glob.glob(pattern):
            with open(fp, encoding="utf-8") as f:
                doc = json.load(f)
            doc["source_type"] = source
            docs.append(doc)
 
    print(f"Loaded {len(docs)} documents")
    total_words = sum(d.get("word_count", 0) for d in docs)
    print(f"Total words: {total_words:,}")
    return docs
 
corpus = load_corpus()

Loaded 237 documents
Total words: 318,815


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — CHUNKING STRATEGY 1: Fixed-size chunking
# Simple baseline — split by word count with overlap
# ─────────────────────────────────────────────────────────────────────────────
 
def fixed_size_chunker(text, chunk_size=256, overlap=32):
    """
    Splits text into fixed-size chunks by word count with a sliding overlap.
 
    Args:
        text       : full document text
        chunk_size : target number of words per chunk
        overlap    : number of words to overlap between consecutive chunks
 
    Returns:
        list of text chunk strings
    """
    words  = text.split()
    chunks = []
    start  = 0
 
    while start < len(words):
        end   = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        if len(chunk.strip()) > 20:
            chunks.append(chunk)
        start += chunk_size - overlap  # slide forward with overlap
 
    return chunks
 
 
def chunk_corpus_fixed(corpus, chunk_size=256, overlap=32):
    chunks = []
    for doc in corpus:
        text   = doc.get("text", "")
        title  = doc.get("title", "Unknown")
        url    = doc.get("url", "")
        source = doc.get("source_type", "unknown")
 
        doc_chunks = fixed_size_chunker(text, chunk_size, overlap)
 
        for i, chunk_text in enumerate(doc_chunks):
            chunks.append({
                "chunk_id":     f"{source}_{title[:30]}_{i}",
                "text":         chunk_text,
                "doc_title":    title,
                "doc_url":      url,
                "source_type":  source,
                "chunk_index":  i,
                "total_chunks": len(doc_chunks),
                "strategy":     "fixed",
                "word_count":   len(chunk_text.split()),
            })
 
    return chunks
 
 
# Run fixed chunking with two different sizes for comparison
chunks_fixed_256 = chunk_corpus_fixed(corpus, chunk_size=256, overlap=32)
chunks_fixed_512 = chunk_corpus_fixed(corpus, chunk_size=512, overlap=64)
 
print(f"Fixed-256 chunks : {len(chunks_fixed_256):,}")
print(f"Fixed-512 chunks : {len(chunks_fixed_512):,}")
print(f"Avg words (256)  : {np.mean([c['word_count'] for c in chunks_fixed_256]):.1f}")
print(f"Avg words (512)  : {np.mean([c['word_count'] for c in chunks_fixed_512]):.1f}")
 

Fixed-256 chunks : 1,536
Fixed-512 chunks : 827
Avg words (256)  : 234.5
Avg words (512)  : 430.4


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — CHUNKING STRATEGY 2: Sentence-based chunking
# Groups complete sentences into chunks — preserves sentence boundaries
# ─────────────────────────────────────────────────────────────────────────────
 
from nltk.tokenize import sent_tokenize
 
def sentence_chunker(text, target_words=200, overlap_sentences=1):
    """
    Splits text into chunks that respect sentence boundaries.
    Accumulates sentences until target word count is reached.
 
    Args:
        text              : full document text
        target_words      : target chunk size in words
        overlap_sentences : number of sentences to carry over to next chunk
 
    Returns:
        list of text chunk strings
    """
    sentences = sent_tokenize(text)
    chunks    = []
    current   = []
    count     = 0
 
    for sent in sentences:
        words  = sent.split()
        count += len(words)
        current.append(sent)
 
        if count >= target_words:
            chunk_text = " ".join(current)
            if len(chunk_text.strip()) > 20:
                chunks.append(chunk_text)
            # Overlap: keep last N sentences
            current = current[-overlap_sentences:] if overlap_sentences > 0 else []
            count   = sum(len(s.split()) for s in current)
 
    # Don't forget the last partial chunk
    if current:
        chunk_text = " ".join(current)
        if len(chunk_text.strip()) > 20:
            chunks.append(chunk_text)
 
    return chunks
 
 
def chunk_corpus_sentence(corpus, target_words=200, overlap_sentences=1):
    chunks = []
    for doc in corpus:
        text   = doc.get("text", "")
        title  = doc.get("title", "Unknown")
        url    = doc.get("url", "")
        source = doc.get("source_type", "unknown")
 
        doc_chunks = sentence_chunker(text, target_words, overlap_sentences)
 
        for i, chunk_text in enumerate(doc_chunks):
            chunks.append({
                "chunk_id":     f"{source}_{title[:30]}_{i}",
                "text":         chunk_text,
                "doc_title":    title,
                "doc_url":      url,
                "source_type":  source,
                "chunk_index":  i,
                "total_chunks": len(doc_chunks),
                "strategy":     "sentence",
                "word_count":   len(chunk_text.split()),
            })
 
    return chunks
 
 
chunks_sentence = chunk_corpus_sentence(corpus, target_words=200, overlap_sentences=1)
 
print(f"Sentence chunks  : {len(chunks_sentence):,}")
print(f"Avg words        : {np.mean([c['word_count'] for c in chunks_sentence]):.1f}")
print(f"Min words        : {min(c['word_count'] for c in chunks_sentence)}")
print(f"Max words        : {max(c['word_count'] for c in chunks_sentence)}")
 

Sentence chunks  : 1,849
Avg words        : 203.2
Min words        : 12
Max words        : 333


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — CHUNKING STRATEGY 3: Semantic chunking (BEST / most advanced)
# Splits when semantic similarity between adjacent sentences drops sharply.
# This is the state-of-the-art approach used in production RAG systems.
# ─────────────────────────────────────────────────────────────────────────────
 
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
 
# Use a lightweight model just for the chunking similarity computation
print("Loading lightweight model for semantic chunking...")
_chunk_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Chunking model loaded")
 
 
def semantic_chunker(text, threshold=0.45, min_chunk_words=50, max_chunk_words=400):
    """
    Semantic chunking: splits text at points where the cosine similarity
    between adjacent sentence embeddings drops below a threshold.
 
    This preserves topically coherent chunks — a new chunk starts whenever
    the topic shifts. This outperforms fixed-size chunking for retrieval
    because each chunk is semantically self-contained.
 
    Args:
        text            : full document text
        threshold       : cosine similarity below which a split is made (0-1)
        min_chunk_words : do not split if resulting chunk would be too small
        max_chunk_words : force a split if chunk exceeds this size
 
    Returns:
        list of text chunk strings
    """
    sentences = sent_tokenize(text)
 
    # Need at least 2 sentences to compare
    if len(sentences) < 2:
        return [text] if text.strip() else []
 
    # Embed all sentences at once (faster than one by one)
    embeddings = _chunk_model.encode(sentences, batch_size=32, show_progress_bar=False)
 
    # Compute cosine similarity between consecutive sentence pairs
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = sklearn_cosine(
            embeddings[i].reshape(1, -1),
            embeddings[i+1].reshape(1, -1)
        )[0][0]
        similarities.append(float(sim))
 
    # Identify split points
    chunks        = []
    current_sents = [sentences[0]]
    current_words = len(sentences[0].split())
 
    for i, sim in enumerate(similarities):
        next_sent       = sentences[i + 1]
        next_word_count = len(next_sent.split())
 
        should_split = (
            sim < threshold and
            current_words >= min_chunk_words
        ) or (
            current_words + next_word_count > max_chunk_words
        )
 
        if should_split:
            chunks.append(" ".join(current_sents))
            current_sents = [next_sent]
            current_words = next_word_count
        else:
            current_sents.append(next_sent)
            current_words += next_word_count
 
    # Final chunk
    if current_sents:
        chunks.append(" ".join(current_sents))
 
    return [c for c in chunks if len(c.strip()) > 20]
 
 
def chunk_corpus_semantic(corpus, threshold=0.45):
    chunks = []
    for doc_idx, doc in enumerate(corpus):
        text   = doc.get("text", "")
        title  = doc.get("title", "Unknown")
        url    = doc.get("url", "")
        source = doc.get("source_type", "unknown")
 
        if len(text.split()) < 50:
            continue
 
        doc_chunks = semantic_chunker(text, threshold=threshold)
 
        for i, chunk_text in enumerate(doc_chunks):
            chunks.append({
                "chunk_id":     f"{source}_{title[:30]}_{i}",
                "text":         chunk_text,
                "doc_title":    title,
                "doc_url":      url,
                "source_type":  source,
                "chunk_index":  i,
                "total_chunks": len(doc_chunks),
                "strategy":     "semantic",
                "word_count":   len(chunk_text.split()),
            })
 
        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1}/{len(corpus)} docs...")
 
    return chunks
 
 
print("Running semantic chunking (this takes 2-3 minutes)...")
chunks_semantic = chunk_corpus_semantic(corpus, threshold=0.45)
 
print(f"\nSemantic chunks  : {len(chunks_semantic):,}")
print(f"Avg words        : {np.mean([c['word_count'] for c in chunks_semantic]):.1f}")
print(f"Min words        : {min(c['word_count'] for c in chunks_semantic)}")
print(f"Max words        : {max(c['word_count'] for c in chunks_semantic)}")
 

Loading lightweight model for semantic chunking...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Rohan\anaconda3\envs\rag-project\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rohan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunking model loaded
Running semantic chunking (this takes 2-3 minutes)...
  Processed 50/237 docs...
  Processed 100/237 docs...
  Processed 150/237 docs...
  Processed 200/237 docs...

Semantic chunks  : 3,356
Avg words        : 95.1
Min words        : 4
Max words        : 396


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Compare chunking strategies
# This comparison table is perfect for your presentation slides
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "="*65)
print("  CHUNKING STRATEGY COMPARISON")
print("="*65)
print(f"  {'Strategy':<20} {'Chunks':>8} {'Avg words':>10} {'Min':>6} {'Max':>6}")
print("  " + "─"*60)
 
strategies = [
    ("Fixed (256w)",   chunks_fixed_256),
    ("Fixed (512w)",   chunks_fixed_512),
    ("Sentence-based", chunks_sentence),
    ("Semantic",       chunks_semantic),
]
for name, chunks in strategies:
    wcs = [c["word_count"] for c in chunks]
    print(f"  {name:<20} {len(chunks):>8,} {np.mean(wcs):>10.1f} {min(wcs):>6} {max(wcs):>6}")
 
print("="*65)
print("\nSelected strategy: SEMANTIC (best for retrieval quality)")
print("Reason: semantically coherent chunks improve retrieval precision")
print("        each chunk covers one topic rather than arbitrary word count")
 
# Use semantic chunks as the final choice
FINAL_CHUNKS = chunks_semantic
print(f"\nFinal chunk count: {len(FINAL_CHUNKS):,}")
 


  CHUNKING STRATEGY COMPARISON
  Strategy               Chunks  Avg words    Min    Max
  ────────────────────────────────────────────────────────────
  Fixed (256w)            1,536      234.5      5    256
  Fixed (512w)              827      430.4      6    512
  Sentence-based          1,849      203.2     12    333
  Semantic                3,356       95.1      4    396

Selected strategy: SEMANTIC (best for retrieval quality)
Reason: semantically coherent chunks improve retrieval precision
        each chunk covers one topic rather than arbitrary word count

Final chunk count: 3,356


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Save all chunk sets for reference
# ─────────────────────────────────────────────────────────────────────────────
 
def save_chunks(chunks, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(chunks):,} chunks -> {filepath}")
 
save_chunks(chunks_fixed_256, "data/processed/chunks_fixed_256.json")
save_chunks(chunks_fixed_512, "data/processed/chunks_fixed_512.json")
save_chunks(chunks_sentence,  "data/processed/chunks_sentence.json")
save_chunks(chunks_semantic,  "data/processed/chunks_semantic.json")
 

Saved 1,536 chunks -> data/processed/chunks_fixed_256.json
Saved 827 chunks -> data/processed/chunks_fixed_512.json
Saved 1,849 chunks -> data/processed/chunks_sentence.json
Saved 3,356 chunks -> data/processed/chunks_semantic.json


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — EMBEDDING MODEL 1: all-MiniLM-L6-v2 (baseline)
# Fast, lightweight, widely used baseline embedding model
# ─────────────────────────────────────────────────────────────────────────────
 
from sentence_transformers import SentenceTransformer
import time
 
print("Loading embedding models...\n")
 
model_minilm = SentenceTransformer("all-MiniLM-L6-v2")
model_bge    = SentenceTransformer("BAAI/bge-small-en-v1.5")
 
print("Both embedding models loaded")
print(f"  MiniLM dimension : {model_minilm.get_sentence_embedding_dimension()}")
print(f"  BGE dimension    : {model_bge.get_sentence_embedding_dimension()}")
 
 
def embed_chunks(chunks, model, model_name, batch_size=64):
    """
    Embeds all chunks using the given model.
    Returns numpy array of embeddings.
    """
    texts = [c["text"] for c in chunks]
 
    # BGE models work best with a query prefix — add it for consistency
    if "bge" in model_name.lower():
        texts = [f"Represent this sentence: {t}" for t in texts]
 
    print(f"Embedding {len(texts):,} chunks with {model_name}...")
    start = time.time()
 
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,   # normalise for cosine similarity
    )
 
    elapsed = time.time() - start
    print(f"  Done in {elapsed:.1f}s — shape: {embeddings.shape}")
    return embeddings.astype(np.float32)
 
 
# Embed the FINAL chunks with both models for comparison
print("\n--- Embedding with MiniLM ---")
emb_minilm = embed_chunks(FINAL_CHUNKS, model_minilm, "MiniLM")
 
print("\n--- Embedding with BGE ---")
emb_bge = embed_chunks(FINAL_CHUNKS, model_bge, "BGE")
 

Loading embedding models...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Rohan\anaconda3\envs\rag-project\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rohan\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

Both embedding models loaded
  MiniLM dimension : 384
  BGE dimension    : 384

--- Embedding with MiniLM ---
Embedding 3,356 chunks with MiniLM...


Batches:   0%|          | 0/53 [00:00<?, ?it/s]

  Done in 46.1s — shape: (3356, 384)

--- Embedding with BGE ---
Embedding 3,356 chunks with BGE...


Batches:   0%|          | 0/53 [00:00<?, ?it/s]

  Done in 104.5s — shape: (3356, 384)


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Compare embedding models on a sample query
# ─────────────────────────────────────────────────────────────────────────────
 
from sklearn.metrics.pairwise import cosine_similarity
 
def top_k_results(query, embeddings, chunks, model, model_name, k=5, bge=False):
    """Retrieves top-k chunks for a query using cosine similarity."""
    q = f"Represent this sentence: {query}" if bge else query
    q_emb = model.encode([q], normalize_embeddings=True).astype(np.float32)
    sims  = cosine_similarity(q_emb, embeddings)[0]
    top_k = np.argsort(sims)[::-1][:k]
 
    print(f"\n  {model_name} — top {k} results for: '{query}'")
    for rank, idx in enumerate(top_k, 1):
        print(f"  [{rank}] sim={sims[idx]:.3f} | {chunks[idx]['doc_title']}")
        print(f"       {chunks[idx]['text'][:120]}...")
    return top_k, sims
 
# Test both models on sample questions from your benchmark
TEST_QUERIES = [
    "What is the main ingredient in miso soup?",
    "How is kimchi made?",
    "What makes Sichuan cuisine spicy?",
]
 
print("\n" + "="*65)
print("  EMBEDDING MODEL COMPARISON")
print("="*65)
for query in TEST_QUERIES:
    top_k_results(query, emb_minilm, FINAL_CHUNKS, model_minilm, "MiniLM-L6")
    top_k_results(query, emb_bge,    FINAL_CHUNKS, model_bge,    "BGE-small",  bge=True)
 
print("\nSelected embedding model: BGE-small-en-v1.5")
print("Reason: BGE is trained specifically for retrieval tasks and")
print("        consistently outperforms MiniLM on retrieval benchmarks")
 
# Use BGE embeddings for the final index
FINAL_EMBEDDINGS = emb_bge
FINAL_MODEL_NAME = "BAAI/bge-small-en-v1.5"
 


  EMBEDDING MODEL COMPARISON

  MiniLM-L6 — top 5 results for: 'What is the main ingredient in miso soup?'
  [1] sim=0.771 | Miso soup
       The stock might include ingredients such asnegi,carrot,potatoanddaikonradish. In some versions of the dish, chicken stoc...
  [2] sim=0.762 | Cookbook:Miso Soup
       # Cookbook:Miso Soup

## Introduction

Cookbook|Recipes|Ingredients|Equipment|Techniques|Cookbook Disambiguation Pages|R...
  [3] sim=0.755 | Miso soup
       Thusnegiandtofu, a strongly flavored ingredient mixed with a mildly flavored ingredient, are often combined. Ingredients...
  [4] sim=0.731 | Miso
       oryzaeis the most desirable because of several properties, including the fact that it does not produceaflatoxin. ## Stor...
  [5] sim=0.730 | Miso soup
       Miso soup and whitericeare the central dishes of the traditional Japanese breakfast. The soup has been a favorite of com...

  BGE-small — top 5 results for: 'What is the main ingredient in miso soup?'
  [1] sim=0.831

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Build FAISS vector store
# ─────────────────────────────────────────────────────────────────────────────
 
import faiss
 
print(f"\nBuilding FAISS index...")
print(f"  Embeddings shape: {FINAL_EMBEDDINGS.shape}")
print(f"  Dimension       : {FINAL_EMBEDDINGS.shape[1]}")
 
dim = FINAL_EMBEDDINGS.shape[1]
 
# We use IndexFlatIP (inner product) since embeddings are L2-normalised,
# which is equivalent to cosine similarity
index = faiss.IndexFlatIP(dim)
index.add(FINAL_EMBEDDINGS)
 
print(f"  Vectors in index: {index.ntotal:,}")
 
# Quick sanity check — search for a test query
q_text  = "What is miso soup made from?"
q_emb   = model_bge.encode(
    [f"Represent this sentence: {q_text}"],
    normalize_embeddings=True
).astype(np.float32)
 
scores, indices = index.search(q_emb, 3)
print(f"\nSanity check — top 3 for '{q_text}':")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    print(f"  [{rank}] score={score:.4f} | {FINAL_CHUNKS[idx]['doc_title']}")
    print(f"       {FINAL_CHUNKS[idx]['text'][:100]}...")
 


Building FAISS index...
  Embeddings shape: (3356, 384)
  Dimension       : 384
  Vectors in index: 3,356

Sanity check — top 3 for 'What is miso soup made from?':
  [1] score=0.8153 | Miso soup
       # Miso soup

## Introduction

Miso soup(味噌汁 or お味噌汁,miso-shiru or omiso-shiru; お-/o- beinghonorific)...
  [2] score=0.8058 | Miso
       # Miso

## Introduction

Miso(みそor味噌)is a traditionalJapanese seasoning. It is a thick paste produce...
  [3] score=0.7892 | Miso
       In medieval times, the wordtemaemiso, meaning homemade miso, appeared. Miso production is relatively...


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Save vector store to disk
# ─────────────────────────────────────────────────────────────────────────────
 
# Save FAISS index
faiss.write_index(index, "vector_store/index.faiss")
print("Saved: vector_store/index.faiss")
 
# Save chunks alongside (needed to recover text from FAISS search results)
with open("vector_store/chunks.pkl", "wb") as f:
    pickle.dump(FINAL_CHUNKS, f)
print("Saved: vector_store/chunks.pkl")
 
# Save BM25 index for hybrid retrieval (used in notebook 04)
from rank_bm25 import BM25Okapi
 
tokenised = [c["text"].lower().split() for c in FINAL_CHUNKS]
bm25      = BM25Okapi(tokenised)
 
with open("vector_store/bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)
print("Saved: vector_store/bm25.pkl")
 
# Save ingestion metadata
metadata = {
    "created_at":       datetime.now().isoformat(),
    "chunking_strategy":"semantic",
    "embedding_model":  FINAL_MODEL_NAME,
    "embedding_dim":    dim,
    "total_chunks":     index.ntotal,
    "total_docs":       len(corpus),
    "chunking_comparison": {
        "fixed_256":  len(chunks_fixed_256),
        "fixed_512":  len(chunks_fixed_512),
        "sentence":   len(chunks_sentence),
        "semantic":   len(chunks_semantic),
    },
    "embedding_models_compared": ["all-MiniLM-L6-v2", "BAAI/bge-small-en-v1.5"],
    "selected_embedding_model":  FINAL_MODEL_NAME,
    "reason_chunking":  "Semantic chunking preserves topical coherence, improving retrieval precision over fixed-size splitting",
    "reason_embedding": "BGE-small is trained for retrieval tasks and outperforms MiniLM on passage retrieval benchmarks",
}
 
with open("vector_store/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved: vector_store/metadata.json")
 

Saved: vector_store/index.faiss
Saved: vector_store/chunks.pkl
Saved: vector_store/bm25.pkl
Saved: vector_store/metadata.json


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Final summary
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "="*60)
print("  INGESTION PIPELINE COMPLETE")
print("="*60)
print(f"\n  Documents ingested : {len(corpus):,}")
print(f"  Final chunks       : {len(FINAL_CHUNKS):,}")
print(f"  Chunking strategy  : Semantic (threshold=0.45)")
print(f"  Embedding model    : {FINAL_MODEL_NAME}")
print(f"  Embedding dim      : {dim}")
print(f"  FAISS vectors      : {index.ntotal:,}")
print(f"\n  Saved files:")
print(f"    vector_store/index.faiss    (FAISS dense index)")
print(f"    vector_store/chunks.pkl     (chunk text + metadata)")
print(f"    vector_store/bm25.pkl       (BM25 sparse index for hybrid)")
print(f"    vector_store/metadata.json  (ingestion config)")
print("\n  Proceed to 04_inference.ipynb")
print("="*60)
 


  INGESTION PIPELINE COMPLETE

  Documents ingested : 237
  Final chunks       : 3,356
  Chunking strategy  : Semantic (threshold=0.45)
  Embedding model    : BAAI/bge-small-en-v1.5
  Embedding dim      : 384
  FAISS vectors      : 3,356

  Saved files:
    vector_store/index.faiss    (FAISS dense index)
    vector_store/chunks.pkl     (chunk text + metadata)
    vector_store/bm25.pkl       (BM25 sparse index for hybrid)
    vector_store/metadata.json  (ingestion config)

  Proceed to 04_inference.ipynb
